# TP: Spark and RDDs

## Imports and Initialization

In [ ]:
from pyspark import SparkContext, SparkConf

#### Configure and initialize a SparkContext instance
* We create a connection to the local Spark cluster (and start it if necessary)

* The context instance represents the connection to the cluster, and serves as main entry point

In [ ]:
conf = SparkConf().setMaster("local").setAppName("Spark TP")
sc = SparkContext(conf=conf)

## First Steps
### Creating RDDs from a text file


In [ ]:
# Let's create our first RDD 
story = sc.textFile("../data/AliceInWonderLandPart1.txt")
head = story.take(5) # Inspect the first 5 elements of the RDD
head

In [ ]:
type(story)  # Let's verify that story is indeed an RDD

In [ ]:
type(head)  # After applying the take action, we no longer have an RDD, but a value

### Finding the frequency of each word in the text

We follow the example from the lecture.

In [ ]:
words = story.flatMap(lambda line: line.split())  # Split each line into words
words.take(5)

In [ ]:
type(words)  # Datatype after performing a transformation

In [ ]:
word_counter_pairs = words.map(lambda word: (word, 1))  # Map each word occurence to count 1
word_counter_pairs.take(5)

In [ ]:
word_counter_pairs = word_counter_pairs.reduceByKey(lambda c1, c2: c1 + c2)
word_counter_pairs.take(5)

In [ ]:
type(word_counter_pairs.take(5))

### Ordering the words by their frequency, with the most frequent one first

In [ ]:
reverse_count = word_counter_pairs.map(lambda pair: (pair[1], pair[0]))  # Swap the key-value pairs
reverse_count.take(5)

In [ ]:
sort_rev_count = reverse_count.sortByKey(ascending=False)  # Sort the pairs by keys in descending order
sort_rev_count.take(5)

### Counting the number of words in each line

In [ ]:
line_size_rdd = (
    story
    .map(lambda line: (len(line.split()), line))
    .sortByKey(ascending=False)
    .map(lambda pair: (pair[1], pair[0]))
)
line_size_rdd.take(5)

In [ ]:
# Print the line with the maximum word length
line_size_rdd.first()[0]

## Transformations

### Mapping RDDs

#### Some tips
You can use help to find some quick info on a function.
You can also write RDD `variable name + . (dot)` and then press tab for a list of available functions.

In [ ]:
help(story.map)

Note: lambda functions are common in Spark RDDs. If you want to know more with lambda functions, follow
https://www.w3schools.com/python/python_lambda.asp

In [ ]:
# It is possible to use regular functions as well. This can be useful for more complex functions.

def give_length(x):
  return (x, len(x))

story.map(give_length).take(4)

In [ ]:
# flatMap flattens the output to a single level list

story.flatMap(lambda x: (x, len(x))).take(4)

### Filtering RDDs

In [ ]:
story.filter(lambda x: len(x) > 2).take(4)

### Computing the Union and Intersections of RDDs


In [ ]:
# Let's create some more RDDs. These are pair RDDs, as they contain key-value pairs

first_rdd = sc.parallelize((
  (1, "Batman"),
  (2, "Superman"),
  (3, "Spiderman")
))

second_rdd = sc.parallelize((
  (3, "Spiderman"),
  (4, "Hulk"),
  (5, "Iron Man")
))

In [ ]:
first_rdd.union(second_rdd).take(10)

In [ ]:
first_rdd.intersection(second_rdd).take(10)

### Deduplicating RDDs


In [ ]:
# union returns duplicate values for spiderman
first_rdd.union(second_rdd).take(10)

In [ ]:
# returns distinct values
first_rdd.union(second_rdd).distinct().take(10)

# note: in general, the order is not preserved

### ByKey Operations
* groupByKey
* reduceByKey
* sortByKey
* aggregateByKey

Recall: these operations expect pair RDDs containing _(key, value)_ pairs

In [ ]:
for i in first_rdd.union(second_rdd).groupByKey().take(10): 
  print(i[0], [j for j in i[1]])  

#### Take away message
- reduceByKey is generally faster and preferred for grouping, compared to groupByKey
- reduce will perform calculations, eg. aggregations, within partitions, and provides a smaller output


#### Reducing
Functions passed to *ByKey must be **commutative** and **associative** (eg., add and multiply). Average/st deviation is not directly implementable.


In [ ]:
(first_rdd
 .union(second_rdd)
 .reduceByKey(lambda k ,v: (k, v))
 .take(10)
)

In [ ]:
(first_rdd
 .union(second_rdd)
 .map(lambda pair: (pair[1], pair[0]))
 .reduceByKey(lambda v1, v2: v1 + v2)
 .take(10)
)

#### Sorting

In [ ]:
(first_rdd
 .union(second_rdd)
 .sortByKey(ascending=False)
 .take(10)
)

In [ ]:
# sorting in combination with distinct
(first_rdd
 .union(second_rdd)
 .distinct()
 .sortByKey()
 .take(10)
)

#### Aggregating
aggregateByKey takes 3 inputs:
1. An initial value -- a neutral element for aggregation (treated as the first value, **on each partition**)
2. A function to ab applied within a partition
3. A function to be applied to the aggregation results for each partition

In [ ]:
# 

(first_rdd
 .union(second_rdd)
 .aggregateByKey("", lambda acc, v: acc + v, lambda acc_p1, acc_p2: acc_p1 + ", " + acc_p2)
 .take(5)
)

# Play with the parameters passed to aggregateByKey()
# e.g., aggregateBeyKey("***", lambda x, y: x, lambda x, y: x + "--" + y)
# aggregateBeyKey("***", lambda x, y: y, lambda x, y: x + "--" + y)
# aggregateBeyKey("***", lambda x, y: x + "^^^" + y, lambda x, y: x + "--" + y)

In [ ]:
# Let's create another RDD for further calculations
third_rdd = sc.parallelize((
  (10, 'Batman'),
  (20, 'Superman'),
  (30, 'Hulk'),
  (40, 'Hulk')
))

#### Calculating the Average

In [ ]:
(first_rdd
 .union(second_rdd)
 .union(third_rdd)
 .map(lambda pair: (pair[1], pair[0]))
 .aggregateByKey(
     (0, 0),
    lambda acc, v: (acc[0] + v, acc[1] + 1),
    lambda acc_p1, acc_p2: (acc_p1[0] + acc_p2[0], acc_p1[1] + acc_p2[1]),
 )
 # .mapValues(lambda v: v[0] / v[1]) # mapValues works on pair RDDs, only mapping the values part of the key-values
 .take(5)
)

In [ ]:
# some general math functions based on reduce
(first_rdd
 .union(second_rdd)
 .union(third_rdd)
 .map(lambda pair: (pair[1], pair[0]))
# .reduceByKey(lambda v1, v2: v1 + v2)
# .reduceByKey(min)
# .reduceByKey(max)
 .take(10)
)

In [ ]:
# reduceByKey is not intuitively suited for getting the mean (or std deviation, etc). See above for aggregateByKey solution
(first_rdd
 .union(second_rdd)
 .union(third_rdd)
 .map(lambda pair: (pair[1], pair[0]))
 .mapValues(lambda v: (v, 1))
 .reduceByKey(lambda v1, v2: (v1[0] + v2[0], v1[1] + v2[1]))
 .map(lambda pair: (pair[0], pair[1][0] / pair[1][1]))
 .take(10)
)

## Joining RDDs

In [ ]:
# join is by key, returns (key, (value A, value B))
(first_rdd
 .join(second_rdd)
 .take(5)
)

In [ ]:
def swap(pair):
    k, v = pair
    return (v, k)

(first_rdd
 .map(swap)
 .join(third_rdd.map(swap))
 .take(5)
)

# Actions

## Displaying contents


In [ ]:
first_rdd.collect()  # takes full dataset into driver memory

In [ ]:
n = 5
first_rdd.take(n)  # returns a list of n items

In [ ]:
first_rdd.first()  # returns the first element from RDD

In [ ]:
# return type of first() is not same as for take(1)
print("first_rdd.take(1):\t", type(first_rdd.take(1)))
print("first_rdd.first():\t", type(first_rdd.first()))

In [ ]:
first_rdd.top(n)  # returns n items starting from "the top"

In [ ]:
# count of items/rows in RDD
first_rdd.count()

## Reducing RDDs without Keys

In [ ]:
(first_rdd
 # .map(lambda pair: pair[0])
 # .reduce(lambda pair1, pair2: pair1 + pair2)  #sum
 #.reduce(max)
 # .reduce(min)
)

In [ ]:
# simple sum/mean/max/min functions

first_rdd.map(lambda x: x[0]).sum()
#first_rdd.map(lambda x: x[0]).mean()
#first_rdd.map(lambda x: x[0]).min()
#first_rdd.map(lambda x: x[0]).max()

# Exercise Task

The data file `movie_data` is a tsv and has the following format
```
userID movieID rating timestamp
```
The rating ranges from 1 to 5 and timestamp is in the form of a number.

#### Task
Write a Spark Application that computes the average rating per movie. The output should have the format
```
movieID avgRating
```